# Width measures on synthetic scenes and the size of every data set

**Computes:** Alignment ratio and Sambuc index of the fuzzifier-pair and power footprints on four synthetic scenes (eight seeds), and the size of every tabular data set and synthetic instance.

**Inputs:** UCI data (downloaded).  
**Outputs:** `results/16/results_16.json` and a text report.


In [ ]:
import os, sys, time, json, pickle, platform, subprocess, glob, urllib.request
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATASETS = None
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")

RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.join(RESULTS_DIR, "16")
OUTDIR = os.path.abspath(OUTDIR)
for sub in ("data", "reports"):
    os.makedirs(os.path.join(OUTDIR, sub), exist_ok=True)
import numpy as np, scipy, sklearn
print(f"colab {IN_COLAB} | python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")
print(f"output dir: {OUTDIR}   run stamp: {RUN_STAMP}")

In [ ]:
import numpy as np
from itertools import permutations

EPS = 1e-12

def fcm_membership(D, m):
    p = 2.0 / (m - 1.0)
    z = -p * np.log(np.maximum(D, EPS))
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def hr_bounds(D, m1, m2):
    C = D.shape[1]
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    s = 1.0 / (D[:, :, None] / np.maximum(D[:, None, :], EPS)).sum(axis=2)
    cond = s < 1.0 / C
    upper = np.where(cond, U1, U2)
    lower = np.where(cond, U2, U1)
    return lower, upper

def km_interval_1d(x, wl, wu):
    o = np.argsort(x, kind="stable"); xs = x[o]; l = wl[o]; u = wu[o]
    n = len(x)
    cl, cu = np.cumsum(l), np.cumsum(u)
    clx, cux = np.cumsum(l * xs), np.cumsum(u * xs)
    Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
    k = np.arange(n + 1)
    Lk = np.concatenate([[0.0], cl]); Uk = np.concatenate([[0.0], cu])
    Lxk = np.concatenate([[0.0], clx]); Uxk = np.concatenate([[0.0], cux])
    vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
    vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
    kR, kL = int(np.argmax(vR_all)), int(np.argmin(vL_all))
    selR = np.zeros(n, bool); selL = np.zeros(n, bool)
    rank = np.empty(n, int); rank[o] = np.arange(n)
    selR[:] = rank >= kR
    selL[:] = rank < kL
    return float(vL_all[kL]), float(vR_all[kR]), selL, selR

def it2fcm_update(X, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    wl, wu = lower ** mc, upper ** mc
    for j in range(C):
        accL = np.zeros(N); accR = np.zeros(N)
        for l in range(M):
            vL, vR, selL, selR = km_interval_1d(X[:, l], wl[:, j], wu[:, j])
            V[j, l] = 0.5 * (vL + vR)
            accL += np.where(selL, upper[:, j], lower[:, j])
            accR += np.where(selR, upper[:, j], lower[:, j])
        uL[:, j], uR[:, j] = accL / M, accR / M
    return V, uL, uR

def dist(X, V):
    return np.maximum(np.linalg.norm(X[:, None, :] - V[None, :, :], axis=2), EPS)

def run_fcm(X, V0, m, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    U = fcm_membership(dist(X, V), m)
    return V, U.argmax(1)

def run_it2fcm(X, V0, m1, m2, mc=None, iters=300, tol=1e-7):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update(X, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update(X, lo, up, mc)
    hard_hr = (0.5 * (uL + uR)).argmax(1)
    hard_nearest = dist(X, V).argmin(1)
    return V, hard_hr, hard_nearest

def accuracy(labels, pred, C):
    best = 0.0
    for perm in permutations(range(C)):
        best = max(best, float(np.mean(np.array(perm)[pred] == labels)))
    return best

def centre_error(V, centres):
    C = len(centres); best = np.inf
    for perm in permutations(range(C)):
        best = min(best, float(np.mean(np.linalg.norm(V[list(perm)] - centres, axis=1))))
    return best

def init_pp(X, C, rng):
    idx = [rng.integers(len(X))]
    for _ in range(1, C):
        d2 = np.min(((X[:, None, :] - X[idx][None]) ** 2).sum(2), axis=1)
        idx.append(rng.choice(len(X), p=d2 / d2.sum()))
    return X[idx].copy()

def scenario(name, seed):
    r = np.random.default_rng(seed)
    if name == "equal":
        c = np.array([[0.0, 0.0], [4.0, 0.0]]); n = (100, 100); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume":
        c = np.array([[0.0, 0.0], [5.0, 0.0]]); n = (100, 100); half = (2.5, 0.6)
        X = np.vstack([c[k] + r.uniform(-half[k], half[k], size=(n[k], 2)) for k in range(2)])
    elif name == "noise":
        c = np.array([[0.0, 0.0], [0.0, 4.0]]); n = (100, 100)
        X = np.vstack([r.normal(c[0], 1.0, size=(n[0], 2)), r.normal(c[1], 1.0, size=(n[1], 2)),
                       r.normal(c[0], 3.0, size=(40, 2))])
        y = np.r_[np.zeros(n[0]), np.ones(n[1]), np.zeros(40)].astype(int)
        return X, y, c, 2
    elif name == "density":
        c = np.array([[0.0, 0.0], [3.5, 0.0]]); n = (200, 30); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume3":
        c = np.array([[0.0, 0.0], [5.0, 0.0], [2.5, 4.0]]); n = (100, 100, 100); s = (1.4, 0.4, 0.8)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(3)])
    else:
        raise ValueError(name)
    y = np.concatenate([np.full(n[k], k) for k in range(len(n))])
    return X, y, c, len(n)

from scipy.optimize import linear_sum_assignment
def accuracy_h(labels, pred, C):
    M = np.zeros((C, C))
    np.add.at(M, (labels, pred), 1)
    r, c = linear_sum_assignment(-M)
    return float(M[r, c].sum() / len(labels))

def align(V, ref):
    Dm = np.linalg.norm(V[:, None] - ref[None], axis=2)
    r, c = linear_sum_assignment(Dm)
    out = np.zeros_like(ref); out[c] = V[r]
    return out

def centre_error_h(V, centres):
    return float(np.mean(np.linalg.norm(align(V, centres) - centres, axis=1)))

def real_dataset(name):
    from sklearn import datasets
    d = {"iris": datasets.load_iris, "wine": datasets.load_wine, "breast_cancer": datasets.load_breast_cancer}[name]()
    X = d.data.astype(float); X = (X - X.mean(0)) / X.std(0)
    y = d.target.astype(int)
    return X, y, len(np.unique(y))

def run_fcm_decoupled(X, V0, ma, mb, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    return V, dist(X, V).argmin(1)

def it2fcm_update_fast(X, order, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    Xs = np.take_along_axis(X, order, axis=0)
    rank = np.empty_like(order); np.put_along_axis(rank, order, np.arange(N)[:, None].repeat(M, 1), axis=0)
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    z = np.zeros((1, M))
    for j in range(C):
        wl = (lower[:, j] ** mc)[order]; wu = (upper[:, j] ** mc)[order]
        cl, cu = np.cumsum(wl, 0), np.cumsum(wu, 0)
        clx, cux = np.cumsum(wl * Xs, 0), np.cumsum(wu * Xs, 0)
        Lk, Uk = np.vstack([z, cl]), np.vstack([z, cu]); Lxk, Uxk = np.vstack([z, clx]), np.vstack([z, cux])
        Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
        vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
        vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
        kR, kL = vR_all.argmax(0), vL_all.argmin(0)
        vR = vR_all[kR, np.arange(M)]; vL = vL_all[kL, np.arange(M)]
        V[j] = 0.5 * (vL + vR)
        selR = rank >= kR[None, :]; selL = rank < kL[None, :]
        up, lo = upper[:, j][:, None], lower[:, j][:, None]
        uR[:, j] = np.where(selR, up, lo).mean(1); uL[:, j] = np.where(selL, up, lo).mean(1)
    return V, uL, uR

def run_it2fcm_fast(X, V0, m1, m2, mc=None, iters=300, tol=1e-7, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

def real_dataset2(name, data_dir="data"):
    import os, pandas as pd
    if name in ("iris", "wine", "wdbc"):
        return real_dataset({"wdbc": "breast_cancer"}.get(name, name))
    if name == "sonar":
        d = pd.read_csv(os.path.join(data_dir, "sonar.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = (d.iloc[:, -1] == "M").to_numpy(int)
    elif name == "pima":
        d = pd.read_csv(os.path.join(data_dir, "pima-indians-diabetes.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = d.iloc[:, -1].to_numpy(int)
    elif name == "wbc":
        d = pd.read_csv(os.path.join(data_dir, "breast-cancer-wisconsin.data"), header=None, na_values="?").dropna()
        X = d.iloc[:, 1:-1].to_numpy(float); y = (d.iloc[:, -1] == 4).to_numpy(int)
    else:
        raise ValueError(name)
    X = (X - X.mean(0)) / X.std(0)
    return X, y, len(np.unique(y))

EXTRA = {"seeds": "wheat-seeds.csv", "glass": "glass.csv", "ionosphere": "ionosphere.csv",
         "banknote": "banknote_authentication.csv", "ecoli": "ecoli.csv"}
def real_dataset3(name, data_dir="data"):
    import os, pandas as pd
    if name not in EXTRA:
        return real_dataset2(name, data_dir)
    d = pd.read_csv(os.path.join(data_dir, EXTRA[name]), header=None)
    X = d.iloc[:, :-1].to_numpy(float)
    y = pd.factorize(d.iloc[:, -1].astype(str).str.strip(), sort=True)[0]
    X = X[:, X.std(0) > 0]
    X = (X - X.mean(0)) / X.std(0)
    return X, y.astype(int), len(np.unique(y))

In [ ]:
import time
import numpy as np

def run_it2fcm_nt(X, V0, m1, m2, mc=None, iters=300, tol=1e-6):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        W = 0.5 * (lo ** mc + up ** mc)
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    return V, (0.5 * (lo + up)).argmax(1)

def contingency(y, p, C):
    M = np.zeros((int(y.max()) + 1, C)); np.add.at(M, (y, p), 1); return M

def ari_from_cont(M):
    comb = lambda x: x * (x - 1) / 2.0
    n = M.sum(); s_ij = comb(M).sum(); a = comb(M.sum(1)).sum(); b = comb(M.sum(0)).sum()
    exp = a * b / comb(n); mx = 0.5 * (a + b)
    return float((s_ij - exp) / (mx - exp)) if mx != exp else 1.0

def nmi_from_cont(M):
    n = M.sum(); P = M / n; pi = P.sum(1); pj = P.sum(0)
    nz = P > 0
    I = float((P[nz] * np.log(P[nz] / (pi[:, None] * pj[None, :])[nz])).sum())
    H = lambda q: float(-(q[q > 0] * np.log(q[q > 0])).sum())
    d = 0.5 * (H(pi) + H(pj))
    return I / d if d > 0 else 1.0

def acc_from_cont(M):
    from scipy.optimize import linear_sum_assignment
    r, c = linear_sum_assignment(-M); return float(M[r, c].sum() / M.sum())

def scores(y, p, C):
    M = contingency(y, p, C)
    return dict(acc=acc_from_cont(M), ari=ari_from_cont(M), nmi=nmi_from_cont(M))

def silhouette_pre(Dsub, lab):
    ks = np.unique(lab)
    if len(ks) < 2: return -1.0
    n = len(lab); a = np.zeros(n); b = np.full(n, np.inf)
    for k in ks:
        mk = lab == k; cnt = mk.sum()
        s = Dsub[:, mk].sum(1)
        own = lab == k
        a[own] = s[own] / max(cnt - 1, 1)
        other = ~own
        b[other] = np.minimum(b[other], s[other] / cnt)
    sing = np.array([(lab == lab[i]).sum() == 1 for i in range(n)])
    sil = np.where(sing, 0.0, (b - a) / np.maximum(np.maximum(a, b), EPS))
    return float(sil.mean())

def calinski(X, lab):
    ks = np.unique(lab); n, k = len(X), len(ks)
    if k < 2 or k >= n: return -1.0
    mu = X.mean(0); B = W = 0.0
    for c in ks:
        Xc = X[lab == c]; mc_ = Xc.mean(0)
        B += len(Xc) * ((mc_ - mu) ** 2).sum(); W += ((Xc - mc_) ** 2).sum()
    return float((B / (k - 1)) / (W / (n - k))) if W > 0 else -1.0

def noise_instance(frac, seed):
    r = np.random.default_rng(seed); c = np.array([[0.0, 0.0], [0.0, 4.0]]); k = int(round(frac * 100))
    X = np.vstack([r.normal(c[0], 1.0, (100, 2)), r.normal(c[1], 1.0, (100, 2)), r.normal(c[0], 3.0, (k, 2))])
    y = np.r_[np.zeros(100), np.ones(100), np.zeros(k)].astype(int)
    return X, y, 2

def imbalance_instance(C, ratio, seed):
    r = np.random.default_rng(seed)
    R = 3.5 / (2 * np.sin(np.pi / C)); ang = 2 * np.pi * np.arange(C) / C
    c = np.c_[R * np.cos(ang), R * np.sin(ang)]
    n = np.maximum(np.round(100 * ratio ** (-np.arange(C) / max(C - 1, 1))).astype(int), 3)
    X = np.vstack([r.normal(c[k], 1.0, (n[k], 2)) for k in range(C)])
    y = np.concatenate([np.full(n[k], k) for k in range(C)])
    return X, y, C

In [ ]:
from PIL import Image
GRID = (1.05, 1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0)
PAIRS = [(a, b) for i, a in enumerate(GRID) for b in GRID[i + 1:]]
IT2_KEYS = [(a, b, mode) for (a, b) in PAIRS for mode in ("m1", "mid", "m2")]
DEC_KEYS = [(a, b) for a in GRID for b in GRID]
FAMILIES = {"FCM": list(GRID), "IT2FCM": IT2_KEYS, "IT2FCM-NT": IT2_KEYS, "T1dec": DEC_KEYS}
SYN = ("equal", "volume", "noise", "density", "volume3")
REAL = ("iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli")
NOISE = {f"noise{int(f*100):02d}": f for f in (0.0, 0.1, 0.2, 0.4, 0.6)}
IMB = {f"imb_C{C}_r{r}": (C, r) for C in (3, 5, 8) for r in (1, 5, 20)}
N_SYN, N_REAL, N_SWEEP, N_INIT, TOL, N_SIL = 40, 20, 20, 2, 1e-6, 1000

def get_instance(nm, s, data_dir):
    if nm in SYN:
        X, y, c, C = scenario(nm, 1000 + s); return X, y, C
    if nm in NOISE: return noise_instance(NOISE[nm], 6000 + s)
    if nm in IMB: return imbalance_instance(*IMB[nm], 7000 + s)
    X, y, C = real_dataset3(nm, data_dir); r = np.random.default_rng(3000 + s)
    idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], int(round(0.75 * (y == k).sum())), replace=False) for k in range(C)]))
    return X[idx], y[idx], C

def fit(fam, key, X, V0, order):
    if fam == "FCM": return run_fcm(X, V0, key, tol=TOL)
    if fam == "T1dec": return run_fcm_decoupled(X, V0, key[0], key[1], tol=TOL)
    a, b, mode = key; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
    if fam == "IT2FCM":
        V, hp, hn = run_it2fcm_fast(X, V0, a, b, mc=mc, tol=TOL, order=order); return V, hp
    return run_it2fcm_nt(X, V0, a, b, mc=mc, tol=TOL)

def run_instance06(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {f: {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in keys} for f, keys in FAMILIES.items()}
    secs = {f: 0.0 for f in FAMILIES}
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for fam, keys in FAMILIES.items():
            t0 = time.perf_counter()
            for k in keys:
                V, lab = fit(fam, k, X, V0, order)
                sc = evaluate(V, lab) if evaluate else scores(y, lab, C)
                d = rec[fam][k]
                for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
                d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["ch"].append(calinski(X, lab))
            secs[fam] += time.perf_counter() - t0
    return dict(rec=rec, sec_per_run={f: secs[f] / (N_INIT * len(FAMILIES[f])) for f in FAMILIES})

def dibco_instance(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M),
                    f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance06(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out

In [ ]:
DATA_DIR = os.path.join(OUTDIR, "data"); os.makedirs(DATA_DIR, exist_ok=True)
MIRROR = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
UCI = "https://archive.ics.uci.edu/ml/machine-learning-databases/"
EXPECT = {"new_thyroid": (215, 5, 3), "dermatology": (358, 34, 6), "yeast": (1484, 8, 10), "page_blocks": (5473, 10, 5)}

def _get(url, fn):
    p = os.path.join(DATA_DIR, fn)
    if not os.path.exists(p) or os.path.getsize(p) == 0:
        urllib.request.urlretrieve(url, p)
    return p

def _parse(name, how, p):
    import pandas as pd
    if name == "new_thyroid" and how == "mirror":
        d = pd.read_csv(p, header=None); X, y = d.iloc[:, :5], d.iloc[:, 5]
    elif name == "new_thyroid":
        d = pd.read_csv(p, header=None); X, y = d.iloc[:, 1:], d.iloc[:, 0]
    elif name == "dermatology":
        d = pd.read_csv(p, header=None, na_values="?").dropna(); X, y = d.iloc[:, :34], d.iloc[:, 34]
    elif name == "yeast":
        d = pd.read_csv(p, header=None, sep=r"\s+"); X, y = d.iloc[:, 1:9], d.iloc[:, 9]
    elif name == "page_blocks":
        d = pd.read_csv(p, header=None, sep=r"\s+"); X, y = d.iloc[:, :10], d.iloc[:, 10]
    return X.to_numpy(float), y.astype(str).str.strip().to_numpy()

def _openml(name):
    import pandas as pd
    from sklearn.datasets import fetch_openml
    b = fetch_openml(name={"page_blocks": "page-blocks", "new_thyroid": "thyroid-new"}.get(name, name), version=1, as_frame=True, parser="auto")
    X = b.data.apply(pd.to_numeric, errors="coerce"); keep = ~X.isna().any(axis=1)
    return X[keep].to_numpy(float), b.target[keep].astype(str).str.strip().to_numpy()

def load_partb(name):
    cache = os.path.join(DATA_DIR, f"partb_{name}.npz")
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=False); return z["X"], z["y"], int(z["C"])
    tries = {"new_thyroid": [("mirror", MIRROR + "new-thyroid.csv", "new-thyroid.csv"), ("uci", UCI + "thyroid-disease/new-thyroid.data", "new-thyroid.data"), ("openml", None, None)],
             "dermatology": [("uci", UCI + "dermatology/dermatology.data", "dermatology.data"), ("openml", None, None)],
             "yeast": [("uci", UCI + "yeast/yeast.data", "yeast.data"), ("openml", None, None)],
             "page_blocks": [("uciZ", UCI + "page-blocks/page-blocks.data.Z", "page-blocks.data.Z"), ("openml", None, None)]}[name]
    errors = []
    for how, url, fn in tries:
        try:
            if how == "openml":
                Xr, yr = _openml(name)
            elif how == "uciZ":
                pz = _get(url, fn); p = pz[:-2]
                subprocess.run(f"gzip -dc '{pz}' > '{p}'", shell=True, check=True); Xr, yr = _parse(name, how, p)
            else:
                Xr, yr = _parse(name, how, _get(url, fn))
            break
        except Exception as e:
            errors.append(f"{how}: {e!r}"); Xr = None
    if Xr is None:
        raise RuntimeError(f"{name}: every source failed:\n  " + "\n  ".join(errors))
    import pandas as pd
    y = pd.factorize(yr, sort=True)[0].astype(int); C = len(np.unique(y))
    N0, M0, C0 = EXPECT[name]
    assert C == C0 and Xr.shape[1] == M0 and abs(len(Xr) - N0) <= 10, f"{name}: unexpected shape {Xr.shape}, C={C}"
    X = Xr[:, Xr.std(0) > 0]; X = (X - X.mean(0)) / X.std(0)
    np.savez(cache, X=X, y=y, C=C); return X, y, C

In [ ]:
PARTB_SETS = ("new_thyroid", "dermatology", "yeast", "page_blocks")
def inst_any(nm, s):
    if nm in PARTB_SETS:
        X, y, C = load_partb(nm)
        r = np.random.default_rng(3000 + s); f = {"page_blocks": 0.3}.get(nm, 0.75)
        idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], max(2, int(round(f * (y == k).sum()))), replace=False) for k in range(C)]))
        return X[idx], y[idx], C
    return get_instance(nm, s, DATA_DIR)

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter

N = 200; CX = CY = N/2.0; R_TRUE = 60.0
def make_image(rng, noise_sigma=8.0, blur_max=9.0, blur_min=0.6, contrast=140.0):
    yy, xx = np.mgrid[0:N, 0:N].astype(float)
    ang = np.arctan2(yy-CY, xx-CX); rad = np.hypot(xx-CX, yy-CY)
    hard = (rad < R_TRUE).astype(float)
    sig = np.linspace(blur_min, blur_max, 12)
    stack = np.stack([gaussian_filter(hard, s) for s in sig])
    t = np.abs(ang)/np.pi
    idx = t*(len(sig)-1); lo = np.floor(idx).astype(int)
    hi = np.minimum(lo+1, len(sig)-1); fr = idx-lo
    iy = yy.astype(int); ix = xx.astype(int)
    soft = stack[lo, iy, ix]*(1-fr) + stack[hi, iy, ix]*fr
    img = 60 + contrast*soft + rng.normal(0, noise_sigma, soft.shape)
    return np.clip(img, 0, 255), t

YY, XX = np.mgrid[0:N, 0:N].astype(float)

def add_bias(img, s=0.45):
    return np.clip(img*(1.0 + s*((XX/(N-1))-0.5)*2), 0, 255)

def scene_plain(rng, noise=8.0):
    im, _ = make_image(rng, noise_sigma=noise)
    return im, np.hypot(XX-CX, YY-CY) < R_TRUE

def scene_bias(rng, noise=8.0):
    im, gt = scene_plain(rng, noise)
    return add_bias(im), gt

def scene_concave(rng, noise=8.0, n_lobe=5, depth=0.42):
    ang = np.arctan2(YY-CY, XX-CX); rad = np.hypot(XX-CX, YY-CY)
    R = R_TRUE*(1 - depth*np.abs(np.cos(n_lobe*ang/2))**3)
    hard = (rad < R).astype(float); t = np.abs(ang)/np.pi
    sig = np.linspace(0.6, 9.0, 12)
    st = np.stack([gaussian_filter(hard, s) for s in sig])
    idx = t*(len(sig)-1); lo = np.floor(idx).astype(int)
    hi = np.minimum(lo+1, len(sig)-1); fr = idx-lo
    iy, ix = YY.astype(int), XX.astype(int)
    soft = st[lo, iy, ix]*(1-fr) + st[hi, iy, ix]*fr
    return np.clip(60 + 140*soft + rng.normal(0, noise, soft.shape), 0, 255), hard > 0.5

def scene_lowcontrast(rng, noise=8.0):
    im, gt = make_image(rng, noise_sigma=noise, contrast=55.0)
    return im, np.hypot(XX-CX, YY-CY) < R_TRUE

SCENES = [('plain', scene_plain), ('bias', scene_bias),
          ('concave', scene_concave), ('lowcontrast', scene_lowcontrast)]

def it2_memberships(I, v1, v2, m1=1.1, m2=10.1, eps=1e-6):
    d1 = np.abs(I - v1) + eps
    d2 = np.abs(I - v2) + eps
    ratio = d1/d2
    e1 = 2.0/(m1 - 1.0)
    e2 = 2.0/(m2 - 1.0)
    with np.errstate(over='ignore'):
        a = 1.0/(1.0 + np.clip(ratio**e1, 0, 1e12))
        b = 1.0/(1.0 + np.clip(ratio**e2, 0, 1e12))
    u_lo = np.minimum(a, b)
    u_hi = np.maximum(a, b)
    return u_lo, u_hi

def km_centroid(I, w):
    w_lo, w_hi = w
    x = I.ravel(); lo = w_lo.ravel(); hi = w_hi.ravel()
    order = np.argsort(x)
    x = x[order]; lo = lo[order]; hi = hi[order]
    def _km(use_hi_first):
        c = (x*((lo+hi)/2)).sum()/max(((lo+hi)/2).sum(), 1e-9)
        for _ in range(50):
            k = np.searchsorted(x, c)
            k = int(np.clip(k, 1, len(x)-1))
            if use_hi_first:
                wgt = np.concatenate([hi[:k], lo[k:]])
            else:
                wgt = np.concatenate([lo[:k], hi[k:]])
            c_new = (x*wgt).sum()/max(wgt.sum(), 1e-9)
            if abs(c_new - c) < 1e-6: break
            c = c_new
        return float(c)
    return _km(True), _km(False)

def it2fac_variant(img, iters=30, m=2.0, a=0.15, smooth_sigma=1.0, mode='power'):
    I = img.astype(float)
    v1, v2 = np.percentile(I, 85), np.percentile(I, 15)
    u2 = None; hist = []
    for it in range(iters):
        if mode == 'orig':
            u_lo, u_hi = it2_memberships(I, v1, v2)
        else:
            d1 = np.abs(I - v1) + 1e-9; d2 = np.abs(I - v2) + 1e-9
            u = 1.0/(1.0 + np.clip((d1/d2)**(2.0/(m-1.0)), 0, 1e12))
            if mode == 'power':
                p1, p2 = 1.0 - a, 1.0 + a
                u_lo = np.clip(u, 0, 1)**p2; u_hi = np.clip(u, 0, 1)**p1
            else:
                raise ValueError(mode)
        u2 = 0.5*(u_lo + u_hi)
        if smooth_sigma > 0:
            u2 = gaussian_filter(u2, smooth_sigma)
        w1 = (np.clip(u_lo,0,1)**m, np.clip(u_hi,0,1)**m)
        w2 = ((1-np.clip(u_hi,0,1))**m, (1-np.clip(u_lo,0,1))**m)
        c1L, c1R = km_centroid(I, w1); c2L, c2R = km_centroid(I, w2)
        v1n, v2n = 0.5*(c1L+c1R), 0.5*(c2L+c2R)
        hist.append(float((u2**m*(I-v1n)**2).sum() + ((1-u2)**m*(I-v2n)**2).sum()))
        if abs(v1n-v1) < 1e-4 and abs(v2n-v2) < 1e-4:
            v1, v2 = v1n, v2n; break
        v1, v2 = v1n, v2n
    return dict(mask=u2 > 0.5, u2=u2, u_lo=u_lo, u_hi=u_hi, fou=u_hi-u_lo,
                v1=float(v1), v2=float(v2), energy=hist, n_iter=len(hist))

def rho_of(fou, u2, C=2, band=0.02, far=0.25):
    d = np.abs(np.asarray(u2, float) - 1.0 / C); n = d < band; f = d > far
    if n.sum() < 10 or f.sum() < 10: return float("nan")
    return float(np.asarray(fou, float)[n].mean() / max(np.asarray(fou, float)[f].mean(), 1e-12))
SCENE_MEASURES = {}
for name, fn in SCENES:
    eb = {"orig": [], "power": []}; rho = {"orig": [], "power": []}
    for sd in range(8):
        img, gt = fn(np.random.default_rng(sd))
        for mode in ("orig", "power"):
            r = it2fac_variant(img, mode=mode)
            eb[mode].append(float(r["fou"].mean())); rho[mode].append(rho_of(r["fou"], r["u2"]))
    SCENE_MEASURES[name] = dict(rho_pair=float(np.nanmean(rho["orig"])), rho_power=float(np.nanmean(rho["power"])),
                                EB_pair=float(np.mean(eb["orig"])), EB_power=float(np.mean(eb["power"])),
                                EB_pair_by_seed=eb["orig"], EB_power_by_seed=eb["power"],
                                rho_pair_by_seed=rho["orig"], rho_power_by_seed=rho["power"])
    SCENE_MEASURES[name]["inversion"] = SCENE_MEASURES[name]["EB_pair"] / SCENE_MEASURES[name]["EB_power"]

In [ ]:
for _f in ("sonar.csv", "pima-indians-diabetes.csv", "breast-cancer-wisconsin.data", "wheat-seeds.csv", "glass.csv",
           "ionosphere.csv", "banknote_authentication.csv", "ecoli.csv"): _get(MIRROR + _f, _f)
SIZES = {}
for nm in ("iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli") + tuple(PARTB_SETS):
    if nm in PARTB_SETS: X, y = load_partb(nm)[:2]
    else: X, y = real_dataset3(nm, DATA_DIR)[:2]
    sz = np.bincount(np.asarray(y).astype(int)); Xi, yi, Ci = inst_any(nm, 0)
    SIZES[nm] = dict(N=len(y), M=X.shape[1], C=len(sz), largest=int(sz.max()), smallest=int(sz.min()), instance=len(yi))
for nm in ("equal", "volume", "noise", "density", "volume3"):
    X, y, c, C = scenario(nm, 1000); sz = np.bincount(y)
    SIZES[nm] = dict(M=X.shape[1], C=C, largest=int(sz.max()), smallest=int(sz.min()), instance=len(y))
for C in (3, 5, 8):
    SIZES[f"imbalance_C{C}"] = dict(instance_by_ratio={r: len(imbalance_instance(C, r, 7000)[1]) for r in (1, 5, 20)})
SIZES["noise_sweep"] = dict(instance_by_fraction={f: len(noise_instance(f, 6000)[1]) for f in (0.0, 0.1, 0.2, 0.4, 0.6)})

In [ ]:
LINES = []
def say(s=""): print(s); LINES.append(s)
say(f"16 -- width measures on synthetic scenes and the size of every data set   (run {RUN_STAMP})")
for n, v in SCENE_MEASURES.items():
    say(f"{n:12s} rho {v['rho_pair']:.3f} / {v['rho_power']:.3f}   E_B {v['EB_pair']:.4f} / {v['EB_power']:.4f}   ratio {v['inversion']:.1f}"
        f"   (seed 0: E_B {v['EB_pair_by_seed'][0]:.4f} / {v['EB_power_by_seed'][0]:.4f})")
for n, v in SIZES.items(): say(f"{n:14s} {v}")
payload = dict(notebook="16", run_stamp=RUN_STAMP, scenes=SCENE_MEASURES, sizes=SIZES)
for p in (os.path.join(OUTDIR, "results_16.json"), os.path.join(OUTDIR, "results_16_report.txt")):
    with open(p, "w") as fh:
        (json.dump(payload, fh, indent=1) if p.endswith(".json") else fh.write("\n".join(LINES) + "\n")); fh.flush(); os.fsync(fh.fileno())